# 1. Purpose

This notebook combines the completed **Direct** and **Cascaded** speech-to-text translation outputs into one final translated-sample metadata file for downstream evaluation.

Inputs:

- Cascaded predictions: `../runs/cascade_full_run_1788146589/cascade_predictions.csv`
- Direct predictions: `../runs/direct_full_run_1788151795/direct_predictions.csv`

Output:

- `../data/full_translated_sample/full_translated_sample_metadata.csv`

The Cascaded prediction file contains:

- all original frozen sample metadata;
- `asr_transcript`;
- `cascade_translation`.

The Direct prediction file contains:

- the same original frozen sample metadata;
- `direct_translation`.

The combined output preserves the original metadata exactly and appends:

1. `asr_transcript`
2. `cascade_translation`
3. `direct_translation`

The notebook performs **no evaluation and no modification of model outputs**. It only validates that the two prediction files refer to the same 4,200 samples and combines their outputs safely for later WER, chrF++, BLEU, and statistical-analysis notebooks.

The unique sample key used for alignment is `id` (for example, `sample_0001`). `sample_id` is intentionally **not** used as the merge key because it represents speaker identity and may repeat across multiple clips.


# 2. Import Required Libraries

Only lightweight data-processing libraries are required. No model libraries are needed because inference has already been completed.


In [4]:
from pathlib import Path
import hashlib
import json

import pandas as pd

print("Libraries imported successfully.")


Libraries imported successfully.


# 3. File Paths and Expected Dataset Structure

All paths are defined in one place.

The notebook expects the final experiment design:

- **4,200 total samples**
- **7 accent groups**
- **600 samples per accent group**

The combined file is written under `../data/full_translated_sample/`.


In [5]:
PROJECT_ROOT = Path.cwd().resolve()

CASCADE_PATH = (
    PROJECT_ROOT
    / "../runs"
    / "cascade_full_run_1788146589"
    / "cascade_predictions.csv"
)

DIRECT_PATH = (
    PROJECT_ROOT
    / "../runs"
    / "direct_full_run_1788151795"
    / "direct_predictions.csv"
)

OUTPUT_DIR = PROJECT_ROOT / "../data" / "full_translated_sample"
OUTPUT_PATH = OUTPUT_DIR / "full_translated_sample_metadata.csv"

MERGE_KEY = "id"

EXPECTED_TOTAL_SAMPLES = 4200
EXPECTED_ACCENT_GROUPS = 7
EXPECTED_SAMPLES_PER_ACCENT = 600

print(f"Project root: {PROJECT_ROOT}")
print(f"Cascade input: {CASCADE_PATH}")
print(f"Direct input: {DIRECT_PATH}")
print(f"Output: {OUTPUT_PATH}")


Project root: D:\projects\cs760-accent-st-robustness-ml-research\notebooks
Cascade input: D:\projects\cs760-accent-st-robustness-ml-research\notebooks\..\runs\cascade_full_run_1788146589\cascade_predictions.csv
Direct input: D:\projects\cs760-accent-st-robustness-ml-research\notebooks\..\runs\direct_full_run_1788151795\direct_predictions.csv
Output: D:\projects\cs760-accent-st-robustness-ml-research\notebooks\..\data\full_translated_sample\full_translated_sample_metadata.csv


# 4. Validate Input Files

Check that both prediction files exist before loading them.


In [6]:
if not CASCADE_PATH.is_file():
    raise FileNotFoundError(f"Cascade predictions not found: {CASCADE_PATH}")

if not DIRECT_PATH.is_file():
    raise FileNotFoundError(f"Direct predictions not found: {DIRECT_PATH}")

print("Both input prediction files exist.")


Both input prediction files exist.


# 5. Load Prediction Files

Load the completed Direct and Cascaded prediction CSV files and display their basic structure.


In [7]:
cascade_df = pd.read_csv(CASCADE_PATH)
direct_df = pd.read_csv(DIRECT_PATH)

print(f"Cascade shape: {cascade_df.shape}")
print(f"Direct shape: {direct_df.shape}")

print("\nCascade columns:")
print(cascade_df.columns.tolist())

print("\nDirect columns:")
print(direct_df.columns.tolist())


Cascade shape: (4200, 24)
Direct shape: (4200, 23)

Cascade columns:
['id', 'path', 'sentence_id', 'sentence', 'sentence_domain', 'up_votes', 'down_votes', 'age', 'gender', 'accents', 'variant', 'locale', 'segment', 'primary_accent', 'sentence_norm', 'clip', 'duration[ms]', 'sentence_len_words', 'max_plausible_words', 'sentence_len_chars', 'sample_id', 'reference_translation_zh', 'asr_transcript', 'cascade_translation']

Direct columns:
['id', 'path', 'sentence_id', 'sentence', 'sentence_domain', 'up_votes', 'down_votes', 'age', 'gender', 'accents', 'variant', 'locale', 'segment', 'primary_accent', 'sentence_norm', 'clip', 'duration[ms]', 'sentence_len_words', 'max_plausible_words', 'sentence_len_chars', 'sample_id', 'reference_translation_zh', 'direct_translation']


# 6. Validate Required Prediction Columns

The Cascaded file must contain:

- `asr_transcript`
- `cascade_translation`

The Direct file must contain:

- `direct_translation`

The merge key `id` must also exist in both files.


In [8]:
required_cascade_columns = {
    MERGE_KEY,
    "asr_transcript",
    "cascade_translation",
}

required_direct_columns = {
    MERGE_KEY,
    "direct_translation",
}

missing_cascade = required_cascade_columns - set(cascade_df.columns)
missing_direct = required_direct_columns - set(direct_df.columns)

if missing_cascade:
    raise KeyError(
        f"Cascade predictions are missing required columns: {sorted(missing_cascade)}"
    )

if missing_direct:
    raise KeyError(
        f"Direct predictions are missing required columns: {sorted(missing_direct)}"
    )

print("Required prediction columns are present.")


Required prediction columns are present.


# 7. Validate Sample Identity and Dataset Balance

Before combining the systems, verify that:

- both files contain exactly 4,200 rows;
- `id` is complete and unique in both files;
- both files contain exactly the same set of `id` values;
- the Cascaded dataset contains 7 accent groups × 600 samples.

Using `id` rather than `sample_id` is important because `sample_id` represents speaker identity and can legitimately repeat across clips.


In [9]:
for name, df in [
    ("Cascade", cascade_df),
    ("Direct", direct_df),
]:
    if len(df) != EXPECTED_TOTAL_SAMPLES:
        raise ValueError(
            f"{name} file must contain exactly {EXPECTED_TOTAL_SAMPLES:,} rows, "
            f"but found {len(df):,}."
        )

    if df[MERGE_KEY].isna().any():
        raise ValueError(f"{name} file contains missing {MERGE_KEY!r} values.")

    if df[MERGE_KEY].astype(str).str.strip().eq("").any():
        raise ValueError(f"{name} file contains blank {MERGE_KEY!r} values.")

    if not df[MERGE_KEY].is_unique:
        duplicates = (
            df.loc[df[MERGE_KEY].duplicated(keep=False), MERGE_KEY]
            .head(20)
            .tolist()
        )
        raise ValueError(
            f"{name} file requires unique {MERGE_KEY!r}. "
            f"Example duplicates: {duplicates}"
        )

cascade_ids = set(cascade_df[MERGE_KEY])
direct_ids = set(direct_df[MERGE_KEY])

if cascade_ids != direct_ids:
    only_cascade = list(cascade_ids - direct_ids)[:20]
    only_direct = list(direct_ids - cascade_ids)[:20]
    raise ValueError(
        "Direct and Cascade files do not contain the same sample IDs.\n"
        f"Only in Cascade (examples): {only_cascade}\n"
        f"Only in Direct (examples): {only_direct}"
    )

if "primary_accent" not in cascade_df.columns:
    raise KeyError("Column 'primary_accent' is required for final dataset validation.")

accent_counts = cascade_df["primary_accent"].value_counts(dropna=False)

if len(accent_counts) != EXPECTED_ACCENT_GROUPS:
    raise ValueError(
        f"Expected {EXPECTED_ACCENT_GROUPS} accent groups, "
        f"but found {len(accent_counts)}."
    )

if not accent_counts.eq(EXPECTED_SAMPLES_PER_ACCENT).all():
    raise ValueError(
        "Final dataset is not balanced at "
        f"{EXPECTED_SAMPLES_PER_ACCENT} samples per accent:\n{accent_counts}"
    )

print(f"Total samples: {len(cascade_df):,}")
print(f"Unique {MERGE_KEY}: {cascade_df[MERGE_KEY].nunique():,}")
print("Direct and Cascade contain the same sample IDs.")
print("\nAccent counts:")
display(accent_counts.rename("count").to_frame())


Total samples: 4,200
Unique id: 4,200
Direct and Cascade contain the same sample IDs.

Accent counts:


,count
primary_accent,
England English,600
Filipino,600
Hong Kong English,600
"India and South Asia (India, Pakistan, Sri Lanka)",600
Malaysian English,600
"Southern African (South Africa, Zimbabwe, Namibia)",600
United States English,600


# 8. Validate Shared Metadata

The two inference pipelines should have processed the **same frozen input metadata**.

This section compares every column shared between the two files except model-output columns. The notebook stops if any shared metadata value differs after alignment by `id`.

This prevents accidental combination of predictions generated from different sample manifests or differently ordered datasets.


In [10]:
OUTPUT_COLUMNS = {
    "asr_transcript",
    "cascade_translation",
    "direct_translation",
}

shared_metadata_columns = [
    column
    for column in cascade_df.columns
    if column in direct_df.columns
    and column not in OUTPUT_COLUMNS
]

cascade_aligned = (
    cascade_df
    .set_index(MERGE_KEY, drop=False)
    .loc[direct_df[MERGE_KEY]]
    .reset_index(drop=True)
)

direct_aligned = direct_df.reset_index(drop=True)

metadata_mismatches = []

for column in shared_metadata_columns:
    left = cascade_aligned[column]
    right = direct_aligned[column]

    equal_mask = (
        left.eq(right)
        | (left.isna() & right.isna())
    )

    if not equal_mask.all():
        metadata_mismatches.append(
            {
                "column": column,
                "mismatch_count": int((~equal_mask).sum()),
            }
        )

if metadata_mismatches:
    mismatch_df = pd.DataFrame(metadata_mismatches)
    display(mismatch_df)
    raise ValueError(
        "Shared metadata differs between Direct and Cascade prediction files. "
        "Do not combine until the mismatch is resolved."
    )

print(
    f"All {len(shared_metadata_columns)} shared metadata columns match "
    "exactly after alignment by 'id'."
)


All 22 shared metadata columns match exactly after alignment by 'id'.


# 9. Validate Prediction Completeness

Every final sample should contain:

- a Whisper ASR transcript;
- a Cascaded Chinese translation;
- a Direct Chinese translation.

The notebook does not judge whether the predictions are *correct* here; it only checks that the inference outputs are present.


In [11]:
prediction_checks = {
    "asr_transcript": cascade_aligned["asr_transcript"],
    "cascade_translation": cascade_aligned["cascade_translation"],
    "direct_translation": direct_aligned["direct_translation"],
}

for name, series in prediction_checks.items():
    missing_count = int(series.isna().sum())
    blank_count = int(series.fillna("").astype(str).str.strip().eq("").sum())

    print(
        f"{name}: "
        f"missing={missing_count:,}, "
        f"blank={blank_count:,}"
    )

    if missing_count or blank_count:
        raise ValueError(
            f"{name} is incomplete: "
            f"{missing_count} missing and {blank_count} blank values."
        )

print("All three prediction fields are complete for all samples.")


asr_transcript: missing=0, blank=0
cascade_translation: missing=0, blank=0
direct_translation: missing=0, blank=0
All three prediction fields are complete for all samples.


# 10. Combine Direct and Cascaded Outputs

Use the Cascaded file as the base because it already contains:

- the original frozen metadata;
- `asr_transcript`;
- `cascade_translation`.

Then append the `direct_translation` column by matching the unique `id`.

The original row order is preserved.


In [12]:
combined_df = cascade_df.copy(deep=True)

direct_translation_by_id = (
    direct_df
    .set_index(MERGE_KEY)["direct_translation"]
)

combined_df["direct_translation"] = (
    combined_df[MERGE_KEY]
    .map(direct_translation_by_id)
)

expected_columns = (
    [
        column
        for column in cascade_df.columns
    ]
    + ["direct_translation"]
)

combined_df = combined_df[expected_columns]

print(f"Combined shape: {combined_df.shape}")
print("\nCombined columns:")
print(combined_df.columns.tolist())

display(
    combined_df[
        [
            "id",
            "primary_accent",
            "sentence",
            "reference_translation_zh",
            "asr_transcript",
            "cascade_translation",
            "direct_translation",
        ]
    ].head()
)


Combined shape: (4200, 25)

Combined columns:
['id', 'path', 'sentence_id', 'sentence', 'sentence_domain', 'up_votes', 'down_votes', 'age', 'gender', 'accents', 'variant', 'locale', 'segment', 'primary_accent', 'sentence_norm', 'clip', 'duration[ms]', 'sentence_len_words', 'max_plausible_words', 'sentence_len_chars', 'sample_id', 'reference_translation_zh', 'asr_transcript', 'cascade_translation', 'direct_translation']


,id,primary_accent,sentence,reference_translation_zh,asr_transcript,cascade_translation,direct_translation
0,sample_0001,England English,We talked of the side show in the circus.,我们谈到了马戏团的杂耍表演。,We talked of the side shoe in the circus.,我们谈论了马戏团的侧鞋.,我们谈到了马戏团的副表演.
1,sample_0002,England English,"However, after persistent calls, Hall-Jones re...",然而，经过不断的电话，Hall-Jones 还是不情愿地接受了，尽管他在议会里毫无野心。,"However, after persistent calls, Hall-Jones re...","然而,在不断的呼叫之后,霍尔-乔恩斯不愿意接受,尽管没有议会的野心.","然而,在持续的呼叫之后,霍尔-<unk>斯不情愿地接受了,尽管没有议会野心."
2,sample_0003,England English,What should I say?,我该说什么？,What should I say?,我应该怎么说?,我应该说些什么?
3,sample_0004,England English,The rights to the film are currently in the pu...,这部电影的版权目前属于公共领域。,The rights to the film are currently in the pu...,电影的权利目前是公共领域.,这部电影的版权目前在公共领域.
4,sample_0005,England English,Skiiers on a snowy mountain.,雪山上的滑雪者。,Skiers on a snowy mountain.,在一个雪山上滑雪者.,雪山上的滑雪者


# 11. Final Validation

Before saving, verify that the combined dataset still represents exactly the same frozen 4,200-sample experiment.

Checks include:

- 4,200 rows;
- unique `id`;
- 7 accent groups × 600 samples;
- original Cascaded metadata/output columns preserved;
- `direct_translation` appended once;
- no missing/blank prediction values;
- row order unchanged from the Cascaded predictions;
- Direct translations correctly aligned by `id`.


In [13]:
final_checks = {
    "row_count_is_4200": len(combined_df) == EXPECTED_TOTAL_SAMPLES,
    "id_is_unique": combined_df[MERGE_KEY].is_unique,
    "id_order_preserved": combined_df[MERGE_KEY].equals(cascade_df[MERGE_KEY]),
    "column_order_correct": combined_df.columns.tolist() == expected_columns,
    "accent_group_count_is_7": combined_df["primary_accent"].nunique() == EXPECTED_ACCENT_GROUPS,
    "each_accent_has_600": combined_df["primary_accent"].value_counts().eq(
        EXPECTED_SAMPLES_PER_ACCENT
    ).all(),
    "asr_complete": combined_df["asr_transcript"].fillna("").astype(str).str.strip().ne("").all(),
    "cascade_translation_complete": combined_df["cascade_translation"].fillna("").astype(str).str.strip().ne("").all(),
    "direct_translation_complete": combined_df["direct_translation"].fillna("").astype(str).str.strip().ne("").all(),
    "direct_translation_alignment_correct": combined_df["direct_translation"].equals(
        combined_df[MERGE_KEY].map(direct_translation_by_id)
    ),
}

validation_df = pd.Series(final_checks, name="passed").to_frame()
display(validation_df)

if not all(final_checks.values()):
    failed_checks = [
        name
        for name, passed in final_checks.items()
        if not passed
    ]
    raise AssertionError(
        f"Final validation failed: {failed_checks}"
    )

print("All final validation checks passed.")


,passed
row_count_is_4200,True
id_is_unique,True
id_order_preserved,True
column_order_correct,True
accent_group_count_is_7,True
each_accent_has_600,True
asr_complete,True
cascade_translation_complete,True
direct_translation_complete,True
direct_translation_alignment_correct,True


All final validation checks passed.


# 12. Save Combined Metadata

Create the output directory if needed and save the validated combined metadata to:

```text
../data/full_translated_sample/full_translated_sample_metadata.csv
```

The input prediction files are never modified.


In [14]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

combined_df.to_csv(
    OUTPUT_PATH,
    index=False,
    encoding="utf-8",
)

print(f"Saved combined metadata to:\n{OUTPUT_PATH.resolve()}")


Saved combined metadata to:
D:\projects\cs760-accent-st-robustness-ml-research\data\full_translated_sample\full_translated_sample_metadata.csv


# 13. Verify Saved File

Reload the saved CSV and confirm that its row count, column order, IDs, and prediction fields are unchanged after serialization.


In [15]:
saved_df = pd.read_csv(OUTPUT_PATH)

save_checks = {
    "saved_file_exists": OUTPUT_PATH.is_file(),
    "saved_row_count_matches": len(saved_df) == len(combined_df),
    "saved_columns_match": saved_df.columns.tolist() == combined_df.columns.tolist(),
    "saved_id_order_matches": saved_df[MERGE_KEY].equals(combined_df[MERGE_KEY]),
    "saved_asr_complete": saved_df["asr_transcript"].fillna("").astype(str).str.strip().ne("").all(),
    "saved_cascade_complete": saved_df["cascade_translation"].fillna("").astype(str).str.strip().ne("").all(),
    "saved_direct_complete": saved_df["direct_translation"].fillna("").astype(str).str.strip().ne("").all(),
}

display(pd.Series(save_checks, name="passed").to_frame())

if not all(save_checks.values()):
    failed_checks = [
        name
        for name, passed in save_checks.items()
        if not passed
    ]
    raise AssertionError(
        f"Saved-file validation failed: {failed_checks}"
    )

print("\nCOMBINATION COMPLETE")
print("=" * 70)
print(f"Samples: {len(saved_df):,}")
print(f"Accent groups: {saved_df['primary_accent'].nunique()}")
print("Prediction fields:")
print("  - asr_transcript")
print("  - cascade_translation")
print("  - direct_translation")
print(f"\nOutput:\n{OUTPUT_PATH.resolve()}")


,passed
saved_file_exists,True
saved_row_count_matches,True
saved_columns_match,True
saved_id_order_matches,True
saved_asr_complete,True
saved_cascade_complete,True
saved_direct_complete,True



COMBINATION COMPLETE
Samples: 4,200
Accent groups: 7
Prediction fields:
  - asr_transcript
  - cascade_translation
  - direct_translation

Output:
D:\projects\cs760-accent-st-robustness-ml-research\data\full_translated_sample\full_translated_sample_metadata.csv
